# Pilot 01: human-shaped transfer in LIBERO

This is an **unvalidated hybrid controller experiment**, not a VLA policy or a purely human-derived demonstration. The robot expert supplies approach/grasp actions, transfer endpoints and orientation. Your phone recording supplies the transfer path deviations and duration. This bypasses an unmeasured hand-marker-to-grasp offset; it does not estimate that offset.

Run sequentially in a Colab Linux runtime. Review trajectory plots before replay. Keep this experiment separate from future training data until the grasp, transport and release succeed. Outputs retain provenance.


In [ ]:
# Use the same working LIBERO installation as notebook 01 when available.
import importlib.util
if importlib.util.find_spec("libero") is None:
    %pip install -q "lerobot[libero]"
%pip install -q scipy matplotlib h5py huggingface_hub "imageio[ffmpeg]"

import os
os.environ["MUJOCO_GL"] = "egl"  # Before simulator imports; restart if previously imported otherwise.
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import json, numpy as np

project = Path("/content/drive/MyDrive/Projects/humanoid-challenge")
cfg = json.loads((project / "configs/pilot_01.json").read_text())
output = project / "outputs/pilot_01_replay"
output.mkdir(parents=True, exist_ok=True)


## 1. Review the extracted motion
The extracted data are already available in `data/processed/pilot_01`. To regenerate, run `scripts/process_pilot.py --project <project path>` in an environment with NumPy and OpenCV aruco. Do not install a different OpenCV into a working simulator runtime just for this.

The provisional human transfer interval is 5.0–11.5 s. Grasp and release are manually estimated, not detected from fingers. Inspect the video and adjust these times if needed. Height is measured at the marker, not the bowl.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Video, display

trajectory = np.genfromtxt(project / "data/processed/pilot_01/marker_smoothed.csv", delimiter=",", names=True)
raw = np.genfromtxt(project / "data/processed/pilot_01/marker_raw.csv", delimiter=",", names=True)
print((project / "data/processed/pilot_01/tracking_report.json").read_text())

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

for ax, key in zip(axes, ["x_m", "y_m", "height_m"]):
    ax.plot(raw["seconds"], raw[key], alpha=.7, label="raw marker")
    ax.plot(trajectory["seconds"], trajectory[key], alpha=0.8, label="smoothed")
    ax.axvline(cfg["transfer_start_s"], color="green", linestyle="--")
    ax.axvline(cfg["transfer_end_s"], color="red", linestyle="--")
    ax.set_ylabel(key); ax.grid(); ax.legend()

axes[-1].set_xlabel("Video seconds")
plt.tight_layout()
plt.show()

display(Video(str(project / cfg["video"]), embed=True))


## 2. Load the same task and robot reference
We use `demo_0`, whose action replay succeeded in notebook 01. The dataset revision is saved for provenance. Robot reference states supply pose anchors; during the hybrid rollout we restore only the initial state, then use actions.


In [ ]:
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from huggingface_hub import HfApi, hf_hub_download
import h5py
from scipy.spatial.transform import Rotation, Slerp

name = json.loads((project / "configs/task.json").read_text())["task_name"]
suite = benchmark.get_benchmark_dict()["libero_spatial"]()
task_id = next(i for i in range(suite.n_tasks) if suite.get_task(i).name == name)
task = suite.get_task(task_id)

repo = "yifengzhu-hf/LIBERO-datasets"
api = HfApi()
revision = api.dataset_info(repo).sha
matches = [f for f in api.list_repo_files(repo, repo_type="dataset", revision=revision)
           if f.startswith("libero_spatial/") and name in f and f.endswith(".hdf5")]
assert len(matches) == 1

path = hf_hub_download(repo, matches[0], repo_type="dataset", revision=revision)
with h5py.File(path, "r") as f:
    actions = f["data/demo_0/actions"][:]
    states = f["data/demo_0/states"][:]

env = OffScreenRenderEnv(
    bddl_file_name=os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file),
    camera_heights=256,
    camera_widths=256,
    horizon=2000,
    ignore_done=True
    )
env.seed(0)
env.reset()
assert states.shape[1] == len(env.get_sim_state())

ref_pos, ref_quat = [], []

for state in states:
    observation = env.set_init_state(state)
    ref_pos.append(observation["robot0_eef_pos"].copy())
    ref_quat.append(observation["robot0_eef_quat"].copy())

ref_pos, ref_quat = np.array(ref_pos), np.array(ref_quat)

plt.figure(figsize=(10,3)); plt.plot(actions[:,-1]); plt.xlabel("Reference action index")
plt.ylabel("Gripper: +close / -open")
plt.grid()
plt.show()


## 3. Select robot transfer anchors
The heuristic below selects the longest positive-gripper interval. Verify the endpoint images: the first should show a secured bowl, and the second a bowl over/on the plate just before release. Adjust `robot_start` and `robot_end` if needed. A wrong anchor can invalidate the experiment.


In [ ]:
closed = actions[:,-1] > 0
padded = np.r_[False, closed, False].astype(int)

starts = np.flatnonzero(np.diff(padded) == 1)
ends = np.flatnonzero(np.diff(padded) == -1)
assert len(starts), "No close-gripper segment found"

j = np.argmax(ends-starts)
robot_start = int(min(starts[j]+9, ends[j]-2))
robot_end = int(ends[j])  # State before first opening action
assert 0 <= robot_start < robot_end < len(states), "Choose anchors manually"

print("Suggested robot anchors:", robot_start, robot_end)

fig, axes = plt.subplots(1,2,figsize=(9,4))

for ax, index in zip(axes, [robot_start, robot_end]):
    ob = env.set_init_state(states[index])
    ax.imshow(ob["agentview_image"][::-1]); ax.set_title(f"Reference state {index}"); ax.axis("off")
plt.show()


## 4. Map human path shape between the robot endpoints
We subtract the human endpoint-to-endpoint straight line, rotate horizontal deviations into the robot transfer direction, and add them to the robot endpoint interpolation. Vertical deviations use the marker's camera-facing height. Scale is limited to avoid magnifying noise. The baseline is a straight-line transfer between the same anchors. This is an explicit geometric heuristic, not a calibrated correspondence between hand and gripper.


In [ ]:
human_start, human_end = cfg["transfer_start_s"], cfg["transfer_end_s"]
assert trajectory["seconds"][0] <= human_start < human_end <= trajectory["seconds"][-1]

control_hz = 20
n = int(round((human_end-human_start)*control_hz))+1
u = np.linspace(0,1,n)
t = human_start + u * (human_end-human_start)
h = np.column_stack([np.interp(t, trajectory["seconds"], trajectory[k]) for k in ["x_m","y_m","height_m"]])
a, b = ref_pos[robot_start], ref_pos[robot_end]
dh, dr = h[-1,:2]-h[0,:2], b[:2]-a[:2]
assert np.linalg.norm(dh) > .02, "Human endpoints too close for stable horizontal alignment"

theta = np.arctan2(dr[1],dr[0])-np.arctan2(dh[1],dh[0])
Rxy = np.array([[np.cos(theta),-np.sin(theta)],[np.sin(theta),np.cos(theta)]])
scale = float(np.clip(np.linalg.norm(dr)/np.linalg.norm(dh), .5, 2.0))
residual = h - ((1-u[:,None])*h[0]+u[:,None]*h[-1])
residual[:,:2] = (residual[:,:2] @ Rxy.T)*scale
baseline = (1-u[:,None])*a+u[:,None]*b
waypoints = baseline + residual
assert np.max(np.linalg.norm(residual,axis=1)) < .20, "Large residual: inspect tracking and phase boundaries"

rotation_targets = Slerp([0,1], Rotation.from_quat(ref_quat[[robot_start,robot_end]]))(u)

fig = plt.figure(figsize=(8,6)); ax = fig.add_subplot(111,projection="3d")
ax.plot(*baseline.T, label="Straight-line control")
ax.plot(*waypoints.T,label="Human-shaped transfer")
ax.plot(*ref_pos[robot_start:robot_end+1].T,label="Robot reference",alpha=.6)
ax.set_xlabel("Robot x (m)"); ax.set_ylabel("Robot y (m)"); ax.set_zlabel("Robot z (m)"); ax.legend(); plt.show()

print("Horizontal scale:", scale, "Transfer duration:", human_end-human_start)
np.savez(output/"mapped_transfer.npz", waypoints=waypoints, baseline=baseline, quaternion_xyzw=rotation_targets.as_quat(), human_times=t)


## 5. Attempt a physics rollout
Run only after inspecting the anchors and plotted path. This cell executes robot expert approach/grasp actions, then tracks the human-shaped transfer with the OSC controller, opens the gripper and retracts. It logs success and tracking error; failed attempts are useful diagnostics, not training demonstrations. No physics state is forced after initialization.

This controller is designed for the OSC_POSE configuration verified in notebook 01 (delta control, translation scale 0.05 m, rotation scale 0.5 rad, 20 Hz). It stops on large sustained tracking errors. That is a debugging guard, not a collision guarantee.


In [ ]:
import imageio.v2 as imageio

controller = env.robots[0].controller_config
assert controller["type"] == "OSC_POSE" and controller["control_delta"]
assert np.allclose(controller["output_max"], [.05,.05,.05,.5,.5,.5])
assert controller["policy_freq"] == control_hz

low, high = env.env.action_spec
assert len(low) == 7

frames = []
logs = []
flags = []
obs = []
acts = []

env.reset()
ob=env.set_init_state(states[0])

def step(action, phase, target=None):
    global ob
    obs.append(ob)
    acts.append(np.clip(action, low, high))
    ob, reward, done, info = env.step(np.clip(action, low, high))
    success=bool(env.check_success())
    flags.append(success)

    error = None if target is None else float(np.linalg.norm(target-ob["robot0_eef_pos"]))
    logs.append(dict(phase=phase,success=success,target_error_m=error,action=np.asarray(action).tolist(),eef=ob["robot0_eef_pos"].tolist()))
    frames.append(np.ascontiguousarray(ob["agentview_image"][::-1]))
    return error

def pose_action(target, rotation, grip):
    current = Rotation.from_quat(ob["robot0_eef_quat"])
    angle = (rotation * current.inv()).as_rotvec()
    return np.r_[np.clip((target-ob["robot0_eef_pos"])/.05,-.3,.3), np.clip(angle/.5,-.3,.3), grip]

aborted=None
try:
    for action in actions[:robot_start]:
        step(action, "expert_approach_grasp")

    if np.linalg.norm(ob["robot0_eef_pos"]-a)>.04:
        raise RuntimeError("Expert prefix drifted from anchor; verify the reference action replay")

    bad = 0
    for target, rotation in zip(waypoints, rotation_targets):
        error = step(pose_action(target,rotation,1.),"human_transfer",target)
        bad = bad+1 if error>.05 else 0

        if bad >= 10:
            raise RuntimeError("Tracking error >5 cm for 0.5 s; inspect video")

    for _ in range(20):
        step(pose_action(b, rotation_targets[-1],1.),"settle_at_destination",b)

    if np.linalg.norm(ob["robot0_eef_pos"]-b)>.025:
        raise RuntimeError("Placement endpoint not reached; withholding release")

    for _ in range(20):
        step(pose_action(b, rotation_targets[-1],-1.),"release",b)

    retract = b + np.array([0,0,.08])
    for _ in range(40):
        step(pose_action(retract, rotation_targets[-1],-1.),"retract",retract)

except RuntimeError as exc:
    aborted = str(exc)

finally:
    if frames:
        imageio.mimsave(output/"hybrid_replay.mp4",frames,fps=control_hz)

    report=dict(method=cfg["method"],
                dataset_revision=revision,
                reference_demo="demo_0",
                robot_start=robot_start,
                robot_end=robot_end,
                human_start_s=human_start,
                human_end_s=human_end,
                scale=scale,
                aborted=aborted,
                ever_success=any(flags), final_success=flags[-1] if flags else False,
                success_last_second=all(flags[-20:]) if len(flags)>=20 else False,
                steps=logs)
    (output/"replay_report.json").write_text(json.dumps(report,indent=2))

print({k:v for k,v in report.items() if k!="steps"})

if frames:
    display(Video(str(output/"hybrid_replay.mp4"),embed=True))


In [ ]:
obs[0], acts[0]